In [18]:
from pathlib import Path
import sys

root = Path.cwd().parents[1]
sys.path.append(str(root))

print(f"Root path: {root}")

Root path: /Users/jonasmeddeb/Projects/homebrew


In [22]:
import subprocess
import os
import signal

for port in [8060, 8061]:
    # Get the PID(s) on the port
    result = subprocess.run(["lsof", "-t", f"-i:{port}"], stdout=subprocess.PIPE, text=True)
    pids = result.stdout.strip().split()
    
    for pid in pids:
        if pid:
            try:
                print(f"Stopping process {pid} on port {port}...")
                os.kill(int(pid), signal.SIGTERM) # Clean kill
            except ProcessLookupError:
                pass

dir = os.path.expanduser("~/Projects/homebrew/template")
process = subprocess.Popen(
    ["task", "dev"],
    cwd=dir,
    stderr=subprocess.STDOUT,
    start_new_session=True
)

Stopping process 29784 on port 8060...
Stopping process 29785 on port 8061...


task: Task "infra" is up to date
task: [css] uv run tailwindcss-extra -i src/presentation/htmx/static/css/input.css -o src/presentation/htmx/static/css/app.css --minify
≈ tailwindcss v4.2.2

/*! 🌼 daisyUI 5.5.19 */
Done in 105ms
task: [dev] uv run dev
[WORKER] [01:41:06] INFO     [MainThread] Logging configured level=INFO json_output=False file_handler=True stream_handler=True
[API] INFO:     Started server process [30275]
[API] INFO:     Waiting for application startup.
[API] [01:41:06] INFO     [MainThread] Logging configured level=INFO json_output=False file_handler=True stream_handler=True
[HTMX] INFO:     Started server process [30276]
[HTMX] INFO:     Waiting for application startup.
[HTMX] [01:41:06] INFO     [MainThread] Logging configured level=INFO json_output=False file_handler=True stream_handler=True
[WORKER] [01:41:06] INFO     [MainThread] Registered event handler topic=heartbeat kind=beat version=1 handler=HeartbeatHandler
[WORKER] [01:41:06] INFO     [MainThread] Start

In [30]:
from infrastructure.config import get_settings

settings = get_settings()
print(f"Settings loaded: {settings.model_dump_json(indent=2)}")

base_url = f"http://{settings.api_host}:{settings.api_port}"

Settings loaded: {
  "env": "development",
  "name": "template",
  "version": "0.1.0",
  "debug": false,
  "api_host": "localhost",
  "api_port": 8060,
  "htmx_host": "localhost",
  "htmx_port": 8061,
  "mcp_host": "localhost",
  "mcp_port": 8062,
  "worker_poll_interval": 3,
  "worker_batch_limit": 100,
  "logging": {
    "level": "INFO",
    "json_output": false,
    "file_handler_enabled": true,
    "file_path_pattern": ".logs/{date}.log",
    "stream_handler_enabled": true,
    "stream_format": "[%(asctime)s] %(levelname)-8s [%(threadName)s] %(message)s",
    "stream_date_format": "%H:%M:%S"
  },
  "database": {
    "provider": "postgresql",
    "host": "localhost",
    "port": 5433,
    "user": "app",
    "password": "**********",
    "database": "app",
    "ssl_mode": null
  },
  "outbox": {
    "default_max_attempts": 3,
    "claim_timeout_seconds": 300,
    "output_channel": "default"
  },
  "heartbeat": {
    "beats": 3,
    "interval": 1.0,
    "max_beats": 60
  },
  "keycloa

In [31]:
import httpx

with httpx.Client(base_url=base_url) as client:
    response = client.get("/health")
    if response.status_code == 200:
        print("✅ API is up and running!")
    else:
        print(f"❌ API health check failed with status code: {response.status_code}")

[API] INFO:     ::1:55516 - "GET /health HTTP/1.1" 200 OK
✅ API is up and running!


In [ ]:
with httpx.Client(base_url=base_url) as client:
    # 1. Pass an empty dict (or actual heartbeat values) to satisfy Pydantic
    response = client.post("/jobs/heartbeat", json={
        "beats": 5,
        "interval": 2.5
    })
    
    # 2. Check for 202 Accepted instead of 200 OK
    if response.status_code == 202:
        print("✅ Job heartbeat is active!")
        print(f"Heartbeat response: {response.json()}")
    else:
        print(f"❌ Job heartbeat check failed with status code: {response.status_code}")
        print(f"Response details: {response.text}")

[API] [01:49:20] INFO     [AnyIO worker thread] Appended outbox job id=539bd875-df16-42c0-9b6b-7fd652578cfa topic=heartbeat kind=beat version=1 idempotency_key=None
[API] INFO:     ::1:55802 - "POST /jobs/heartbeat HTTP/1.1" 202 Accepted
✅ Job heartbeat is active!
Heartbeat response: {'job_id': '539bd875-df16-42c0-9b6b-7fd652578cfa'}


[WORKER] [01:49:22] INFO     [MainThread] Claimed 1 outbox row(s) topic=heartbeat kind=beat version=1
[WORKER] [01:49:22] INFO     [MainThread] Claimed 1 outbox job(s) for topic=heartbeat kind=beat version=1
[WORKER] [01:49:22] INFO     [MainThread] Dispatching outbox job id=539bd875-df16-42c0-9b6b-7fd652578cfa topic=heartbeat kind=beat attempt=1
[WORKER] [01:49:22] INFO     [MainThread] Heartbeat job started id=539bd875-df16-42c0-9b6b-7fd652578cfa beats=5
[WORKER] [01:49:22] INFO     [MainThread] Heartbeat id=539bd875-df16-42c0-9b6b-7fd652578cfa beat=1/5
[WORKER] [01:49:25] INFO     [MainThread] Heartbeat id=539bd875-df16-42c0-9b6b-7fd652578cfa beat=2/5
[WORKER] [01:49:28] INFO     [MainThread] Heartbeat id=539bd875-df16-42c0-9b6b-7fd652578cfa beat=3/5
[WORKER] [01:49:30] INFO     [MainThread] Heartbeat id=539bd875-df16-42c0-9b6b-7fd652578cfa beat=4/5
[WORKER] [01:49:33] INFO     [MainThread] Heartbeat id=539bd875-df16-42c0-9b6b-7fd652578cfa beat=5/5
[WORKER] [01:49:35] INFO     [Main

In [40]:
import json
import websockets

def heartbeat(beats = 5 , interval = 10):
    with httpx.Client(base_url=base_url) as client:
        response = client.post("/jobs/heartbeat", json={
            "beats": beats,
            "interval": interval
        })
        
        if response.status_code == 202:
            response = response.json()
            assert 'job_id' in response
            return response['job_id']
        else:
            print(f"❌ Job heartbeat check failed with status code: {response.status_code}")
            print(f"Response details: {response.text}")


job_id = heartbeat()

URI = f"ws://{settings.api_host}:{settings.api_port}/jobs/ws/{job_id}"
try:
    async with websockets.connect(URI) as websocket:
        async for message in websocket:
            data = json.loads(message)
  
            print(f"[{data.get('status').upper()}] - Updated at: {data.get('finished_at') or data.get('started_at') or 'Just now'}")
            print(json.dumps(data, indent=2))
            print("-" * 40)

except websockets.exceptions.ConnectionClosedOK:
    print("\n🏁 Connection closed cleanly: Job reached a terminal state.")
except Exception as e:
    print(f"\n❌ Error: {e}")

[API] [02:00:01] INFO     [AnyIO worker thread] Appended outbox job id=9aedc65b-3b2a-4943-9848-f9ece2cb1ae2 topic=heartbeat kind=beat version=1 idempotency_key=None
[API] INFO:     ::1:58917 - "POST /jobs/heartbeat HTTP/1.1" 202 Accepted
[API] INFO:     ::1:58918 - "WebSocket /jobs/ws/9aedc65b-3b2a-4943-9848-f9ece2cb1ae2" [accepted]
[API] INFO:     connection open
[PENDING] - Updated at: Just now
{
  "id": "9aedc65b-3b2a-4943-9848-f9ece2cb1ae2",
  "status": "pending",
  "payload": {
    "beats": 5,
    "interval": 10.0
  },
  "error": null,
  "last_error": null,
  "attempts": 0,
  "max_attempts": 3,
  "started_at": null,
  "finished_at": null
}
----------------------------------------
[WORKER] [02:00:02] INFO     [MainThread] Claimed 1 outbox row(s) topic=heartbeat kind=beat version=1
[WORKER] [02:00:02] INFO     [MainThread] Claimed 1 outbox job(s) for topic=heartbeat kind=beat version=1
[WORKER] [02:00:02] INFO     [MainThread] Dispatching outbox job id=9aedc65b-3b2a-4943-9848-f9ece2

In [ ]:
import os
import signal

pgid = os.getpgid(process.pid)
os.killpg(pgid, signal.SIGKILL)